# CELDA 1 — Guía rápida ✨ (GUIA.md)

> <span style="background:#E8F5E9;color:#1B5E20;padding:2px 8px;border-radius:10px;font-weight:700">Misión</span>  
> Convertir un **Excel con celdas faltantes** en un **Excel imputado y confiable**, con foco en **calidad** y **trazabilidad**.  
> Método por defecto: <span style="background:#E3F2FD;color:#0D47A1;padding:0 6px;border-radius:8px;font-weight:600">correlación/modelado</span>.  
> Alternativa: <span style="background:#FFF3E0;color:#E65100;padding:0 6px;border-radius:8px;font-weight:600">modo mixto</span> (**similitud + correlación**) para ampliar cobertura.

---

## 🧭 ¿Qué hace el programa?
- Lee tu **Excel** con datos y **faltantes**.  
- Ejecuta **limpieza básica** y, si aplica, calcula **derivados** (campos auxiliares).  
- Imputa con **modelos por correlación/modelado** y, opcionalmente, **similitud**.  
- Genera un **Excel imputado** listo para revisión.  
  *(Opcional: crea un **JSON** de apoyo para auditoría/visualización).*

---

## ✅ Requisitos mínimos
- Tu archivo **Excel** (ej.: `datos.xlsx`).  
- Librerías instaladas desde `requirements.txt`:

    pip install -r requirements.txt

---

## 🚀 Uso en 3 pasos
1. **Prepará tu archivo** (p. ej., `datos.xlsx`).  
2. **Ejecutá el runner**:

       python helper.py --input datos.xlsx

3. **Revisá la salida**  
   - Se genera un **Excel imputado** (mismo nombre con sufijo o el que indiques).  
   - Si usaste **modo mixto**, también se crea un **JSON** consolidado por celda.

---

## 🎛️ Opciones útiles (sin tecnicismos)
- **Método de imputación**  
  - `--method correlacion` (predeterminado): usa modelos por correlación/modelado.  
  - `--method mixto`: alterna **similitud + correlación** para mejorar cobertura.
- **Último recurso (“sin filtro”)**  
  - `--permitir-sin-filtro`: intenta imputar aun con menos restricciones cuando hay poca información.
- **Calidad mínima (umbrales)**  
  - `--min-muestras` y `--min-unicos`: ajustan **mínimos de datos** exigidos para aceptar un modelo.
- **Ver el proceso (depuración)**  
  - `--debug`: muestra **qué se imputó, qué no y por qué**.

> 💡 Recomendación: empezá con **correlación**. Si quedan faltantes, probá **mixto** y/o activá el **fallback**.

---

## 📦 ¿Qué obtengo?
- **Excel imputado** con ayudas para revisión (comentarios/formato).  
- (**Opcional**) **JSON** consolidado por celda para auditoría/visualización.

---

## 📝 Buenas prácticas
- Usá **nombres de columnas** claros y consistentes.  
- Guardá un **backup** del Excel original.  
- Empezá simple; afiná **umbrales** solo si ves modelos descartados por falta de datos.  
- Activá `--debug` para entender decisiones del sistema.

---

## ❓ FAQ exprés
- **¿Necesito programar?** No: el **runner** (`helper.py`) hace todo.  
- **¿Y si falla el exportador “bonito”?** Hay un **respaldo simple** para no perder la salida.  
- **¿Puedo reintentar con otra config?** Sí: ejecutá de nuevo con otras opciones; no afecta tu original.


In [ ]:
# 🚀 Motor Predictor - Lanzador desde notebooks (modo seguro)
import os, sys
from IPython import get_ipython

BASE_DIR = r"C:/Users/delpi/OneDrive/Tesis/ADRpy-VTOL/ADRpy"
ANALISIS_DIR = os.path.join(BASE_DIR, "analisis")
if ANALISIS_DIR not in sys.path:
    sys.path.insert(0, ANALISIS_DIR)

# Preferimos el Excel de Data si está disponible; sino, intentamos un fallback en Results
ruta_preferida = os.path.join(ANALISIS_DIR, "Data", "Datos_aeronaves.xlsx")
ruta_fallback = os.path.join(ANALISIS_DIR, "Results", "Datos_imputados.xlsx")


def _is_readable(path: str) -> bool:
    try:
        with open(path, "rb"):
            return True
    except Exception as e:
        print(f"ℹ️ No se pudo abrir en modo lectura {path!r}: {e}")
        return False

# Asegurar que la config efectiva tenga la ruta al Excel preferido si existe
try:
    from Modulos.controller import load_effective_config, save_overrides, run_pipeline
    cfg = load_effective_config()
    if _is_readable(ruta_preferida):
        cfg.setdefault("entorno", {}).update({"ruta_excel": ruta_preferida})
    elif _is_readable(ruta_fallback):
        cfg.setdefault("entorno", {}).update({"ruta_excel": ruta_fallback})
    save_overrides(cfg)
    print("▶ Ejecutando pipeline vía controlador (sin main.py)…")
    logs = run_pipeline(cfg)  # usa ejecutar_pipeline del loop
    print(logs)
except Exception as e:
    # Último recurso: evitar ejecutar main.py con argumentos del kernel
    print(f"❌ No se pudo ejecutar run_pipeline de forma segura: {e}")
    print("(Sugerencia: use el panel show_ux y el botón 'Guardar y ejecutar')")

In [2]:
# Panel central de ADRpy (todo vive en scripts; la celda sólo lo invoca)
try:
    from Modulos.ux_notebook_panel import show_ux
except ModuleNotFoundError:
    # Hacemos la celda autocontenida: si no está el path, lo agregamos y reintentamos
    import os, sys
    BASE_DIR = r"C:/Users/delpi/OneDrive/Tesis/ADRpy-VTOL/ADRpy"
    ANALISIS_DIR = os.path.join(BASE_DIR, "analisis")
    if ANALISIS_DIR not in sys.path:
        sys.path.insert(0, ANALISIS_DIR)
    from Modulos.ux_notebook_panel import show_ux

show_ux()

# CELDA 2 — Mapa técnico del sistema 🗺️ (lectura fácil, sin código)

> <span style="background:#FFF3CD;color:#7A5E00;padding:2px 8px;border-radius:10px;font-weight:600">Objetivo</span>  
> Entender **cómo está pensado y distribuido** el proyecto que **imputa** un Excel con celdas faltantes: sus **capas**, **archivos** y el **viaje de los datos**. Útil para mantenimiento, extensiones y diagnóstico — **sin leer código**.

---

## 🧩 Vista general (bloques y flujo)

<span style="background:#E3F2FD;color:#0D47A1;padding:2px 8px;border-radius:10px;font-weight:600">Entrada</span>  
📁 **Excel de origen** (tu dataset con faltantes).

<span style="background:#E8F5E9;color:#1B5E20;padding:2px 8px;border-radius:10px;font-weight:600">Capa 1 · Lanzamiento</span>  
🚀 **helper.py** → recibe parámetros, arma el plan y **orquesta** todo el proceso.

<span style="background:#F1F8E9;color:#33691E;padding:2px 8px;border-radius:10px;font-weight:600">Capa 2 · Datos</span>  
📦 **config_and_loading.py / data_processing.py** → abrir Excel, **limpiar** y **preparar** tabla.  
🧮 **derivados.py** → (si corresponde) calcula **campos derivados** antes de imputar.

<span style="background:#FFF8E1;color:#E65100;padding:2px 8px;border-radius:10px;font-weight:600">Capa 3 · Motores de imputación</span>  
🧠 **imputacion_correlacion.py** → modelos por **correlación/modelado** (predeterminado).  
🧭 **imputation_loop.py** → **modo mixto**: combina **similitud + correlación** y elige el mejor resultado.

<span style="background:#FCE4EC;color:#880E4F;padding:2px 8px;border-radius:10px;font-weight:600">Capa 4 · Reglas y selección</span>  
⚖️ **Umbrales mínimos** (muestras y valores únicos) para aceptar un modelo.  
🛟 **Fallback “sin filtro”** (opcional) como último intento cuando hay poca información.

<span style="background:#EDE7F6;color:#311B92;padding:2px 8px;border-radius:10px;font-weight:600">Capa 5 · Salida y reportes</span>  
📤 **excel_export.py** → **Excel imputado** con ayudas de revisión.  
🗂️ **JSON consolidado** (opcional) por celda para auditoría/visualización.

---

## 🧱 Capas del sistema (qué hace cada una)

### 1) Lanzamiento / Orquestación
- **helper.py**  
  - Punto de entrada.  
  - Lee las **opciones** (método, umbrales, fallback, debug).  
  - Coordina el flujo completo: datos → imputación → exportación.

### 2) Datos (leer, unificar, estructurar)
- **config_and_loading.py / data_processing.py**  
  - Abren el Excel, **sanitizan** datos (tipos, duplicados, vacíos) y construyen la tabla base.  
- **derivados.py**  
  - Calcula **derivados** consistentes (si aplica) para que los modelos tengan mejor información.

### 3) Motores de imputación (predecir faltantes)
- **imputacion_correlacion.py**  
  - Ajusta modelos por **correlación/modelado** para completar las celdas vacías.  
- **imputation_loop.py** (modo **mixto**)  
  - Alterna **similitud** y **correlación**, combina resultados y prioriza el más **confiable**.

### 4) Reglas y selección (cuidar calidad)
- **Umbrales mínimos**  
  - Exigen suficiente **muestra** y **variedad** (valores únicos) para considerar válido un modelo.  
- **Fallback “sin filtro”**  
  - Opción final si los criterios habituales no se cumplen y necesitás mayor cobertura.

### 5) Exportación y reportes (entregar resultados)
- **excel_export.py**  
  - Escribe el **Excel imputado** con comentarios y formatos que facilitan la revisión.  
- **JSON consolidado** (opcional)  
  - Resumen por **celda** para auditoría y herramientas de visualización.

---

## 🔁 Viaje del dato (de tu Excel a la salida)

1. **Excel de entrada** → lectura y **limpieza**.  
2. (Si corresponde) **Derivados** → completar campos calculados útiles.  
3. **Imputación** → modelos por correlación/modelado; si pedís **mixto**, combina con similitud.  
4. **Reglas** → verificación de **umbrales**; si no alcanza, **fallback “sin filtro”**.  
5. **Salida** → **Excel imputado** y, si activaste mixto, **JSON** para auditoría/visualización.

---

## 🧭 Principios de diseño (por qué así)

- <span style="background:#E8F5E9;color:#1B5E20;padding:0 6px;border-radius:8px;font-weight:600">Calidad primero</span>  
  Reglas claras y umbrales antes de aceptar un resultado.
- <span style="background:#E3F2FD;color:#0D47A1;padding:0 6px;border-radius:8px;font-weight:600">Transparencia</span>  
  Salidas legibles y opción de **debug** para entender decisiones.  
- <span style="background:#FFF3E0;color:#E65100;padding:0 6px;border-radius:8px;font-weight:600">Modularidad</span>  
  Cada parte hace una cosa: **datos**, **motores**, **reglas**, **exportación**.  
- <span style="background:#FCE4EC;color:#880E4F;padding:0 6px;border-radius:8px;font-weight:600">Resiliencia</span>  
  Si algo “bonito” falla, existe un **respaldo simple** para no perder resultados.

---

## 🧠 Glosario mínimo

- **Celda**: combinación de claves (p. ej., *Aeronave | Parámetro*) que se analiza.  
- **Similitud**: buscar casos parecidos para proponer un valor.  
- **Correlación/modelado**: relaciones estadísticas usadas para **predecir** faltantes.  
- **Fallback “sin filtro”**: intento final con menos restricciones.  
- **JSON consolidado**: archivo auxiliar con el detalle por celda (útil para auditoría/visualización).

---

## ✅ En resumen

- Tenés un **runner** simple que coordina todo con **pocas decisiones**.  
- El flujo prioriza **calidad y trazabilidad** con reglas claras.  
- La salida es un **Excel listo para uso** y, si querés, un **JSON** para analizar o visualizar después.


In [ ]:
# Debug import for ejecutar_pipeline
import sys, os, importlib, traceback
ANALISIS_DIR = r"C:/Users/delpi/OneDrive/Tesis/ADRpy-VTOL/ADRpy/analisis"
if ANALISIS_DIR not in sys.path:
    sys.path.insert(0, ANALISIS_DIR)
try:
    import Modulos.imputation_loop as il
    importlib.reload(il)
    print("OK: Modulos.imputation_loop imported.")
    print("Has ejecutar_pipeline:", hasattr(il, "ejecutar_pipeline"))
except Exception as e:
    print("IMPORT ERROR:", e)
    traceback.print_exc()

In [ ]:
# Force reload controller and run pipeline safely
import sys, os, importlib
ANALISIS_DIR = r"C:/Users/delpi/OneDrive/Tesis/ADRpy-VTOL/ADRpy/analisis"
if ANALISIS_DIR not in sys.path:
    sys.path.insert(0, ANALISIS_DIR)
import Modulos.controller as ctrl
importlib.reload(ctrl)
print("Reloaded Modulos.controller")
print("Testing _import_pipeline():", ctrl._import_pipeline())
cfg = ctrl.load_effective_config()
# Prefer Excel path
pref = os.path.join(ANALISIS_DIR, "Data", "Datos_aeronaves.xlsx")
if os.path.exists(pref):
    cfg.setdefault("entorno", {})["ruta_excel"] = pref
ctrl.save_overrides(cfg)
logs = ctrl.run_pipeline(cfg)
print(logs)